Boceto del código de detección de contenido +18, machista, racista,  etc.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import pipeline
from bertopic import BERTopic

# =====================================================================
# 1. CARGA DE MODELOS PRE-ENTRENADOS
# =====================================================================
print("Cargando el modelo de detección de toxicidad (esto puede tardar la primera vez)...")
# Usamos un modelo multilingüe excelente para clasificar toxicidad, insultos y obscenidad
clasificador_toxico = pipeline("text-classification", 
                               model="unitary/multilingual-toxic-xlm-roberta", 
                               top_k=None)

# =====================================================================
# 2. FUNCIONES DE PROCESAMIENTO NLP
# =====================================================================
def analizar_letra(letra):
    """
    Pasa una letra por el modelo y devuelve los scores de contenido sensible.
    Recorta a 1500 caracteres para evitar el límite de tokens del modelo.
    """
    try:
        # Si la letra es nula o muy corta, la saltamos
        if not isinstance(letra, str) or len(letra.strip()) < 10:
            return None
            
        letra_recortada = letra[:1500] 
        resultados = clasificador_toxico(letra_recortada)[0]
        return {res['label']: res['score'] for res in resultados}
    except Exception as e:
        return None

def procesar_dataframe_toxicidad(df, col_letra='letra'):
    """
    Aplica el modelo a todo el dataset y crea columnas de análisis.
    """
    print(f"Analizando {len(df)} canciones. ¡Paciencia, el NLP lleva su tiempo!")
    
    # Aplicar el modelo
    df['scores_raw'] = df[col_letra].apply(analizar_letra)
    
    # Extraer puntuaciones específicas (manejo de errores por si alguna falló)
    df['score_toxico'] = df['scores_raw'].apply(lambda x: x.get('toxicity', 0) if isinstance(x, dict) else 0)
    df['score_obsceno'] = df['scores_raw'].apply(lambda x: x.get('obscene', 0) if isinstance(x, dict) else 0)
    df['score_insulto'] = df['scores_raw'].apply(lambda x: x.get('insult', 0) if isinstance(x, dict) else 0)
    
    # Crear etiquetas booleanas (ajusta el 0.6 según veáis los resultados)
    df['es_explicito'] = df['score_obsceno'] > 0.6
    df['es_toxico'] = df['score_toxico'] > 0.6 # Aquí entrarían temas machistas/violentos
    
    return df

# =====================================================================
# 3. FUNCIONES DE VISUALIZACIÓN Y ANÁLISIS CRUZADO
# =====================================================================
def graficar_toxicidad_por_genero(df, col_genero='genero'):
    """
    Genera un gráfico de barras con el % de canciones explícitas por género.
    """
    # Agrupar y calcular la media (que equivale al porcentaje al ser booleanos)
    toxicidad_genero = df.groupby(col_genero)['es_explicito'].mean().reset_index()
    toxicidad_genero['es_explicito'] = toxicidad_genero['es_explicito'] * 100
    toxicidad_genero = toxicidad_genero.sort_values('es_explicito', ascending=False)

    plt.figure(figsize=(12, 7))
    sns.barplot(data=toxicidad_genero, x='es_explicito', y=col_genero, palette='flare')
    
    plt.title('Porcentaje de Canciones con Contenido Explícito (+18) por Género', fontsize=16)
    plt.xlabel('Porcentaje (%)', fontsize=12)
    plt.ylabel('Género Musical', fontsize=12)
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

# =====================================================================
# 4. TOPIC MODELING (BERTopic)
# =====================================================================
def descubrir_temas_explicitos(df, col_letra='letra'):
    """
    Extrae los temas principales SOLO de las canciones marcadas como explícitas/tóxicas.
    """
    print("Iniciando Topic Modeling sobre las canciones +18...")
    
    # Filtramos las canciones conflictivas
    df_explicito = df[(df['es_explicito'] == True) | (df['es_toxico'] == True)]
    letras_maduras = df_explicito[col_letra].dropna().tolist()
    
    if len(letras_maduras) < 10:
        print("No hay suficientes canciones +18 para hacer Topic Modeling.")
        return None
        
    modelo_temas = BERTopic(language="multilingual", calculate_probabilities=True)
    temas, probabilidades = modelo_temas.fit_transform(letras_maduras)
    
    info_temas = modelo_temas.get_topic_info()
    
    print("\n--- TOP TEMAS EN CANCIONES EXPLÍCITAS ---")
    print(info_temas[['Topic', 'Count', 'Name']].head(8))
    
    # Descomentar para ver gráficos interactivos en el entorno (Jupyter/Colab)
    # modelo_temas.visualize_barchart().show()
    
    return modelo_temas

# =====================================================================
# FLUJO DE TRABAJO PRINCIPAL (Para cuando tengas el dataset)
# =====================================================================
if __name__ == "__main__":
    # 1. Aquí cargarás vuestro dataset real generado por el web scraping
    # df = pd.read_csv('vuestro_dataset_canciones.csv')
    
    # --- DATOS SIMULADOS PARA QUE PUEDAS PROBAR EL SCRIPT AHORA ---
    datos_mock = {
        'artista': ['Artista A', 'Artista B', 'Artista C', 'Artista D'],
        'genero': ['Reggaeton', 'Pop', 'Heavy Metal', 'Jazz'],
        'letra': [
            "Mueve ese culo en la discoteca, perreando duro hasta abajo...", 
            "Caminando por la playa, bajo la luz de la luna, te quiero mi amor...",
            "Sangre y destrucción, rompiendo cráneos en la oscuridad total...",
            "Suave brisa de la mañana, un café y tu sonrisa..."
        ]
    }
    df = pd.DataFrame(datos_mock)
    # --------------------------------------------------------------

    # 2. Ejecutar tu pipeline
    print("Iniciando pipeline de la Tarea 2...\n")
    df_procesado = procesar_dataframe_toxicidad(df, col_letra='letra')
    
    # 3. Mostrar resultados en consola
    print("\nResultados del análisis:")
    print(df_procesado[['genero', 'score_toxico', 'score_obsceno', 'es_explicito']])
    
    # 4. Generar el gráfico comparativo
    graficar_toxicidad_por_genero(df_procesado, col_genero='genero')
    
    # 5. Descubrir de qué hablan las canciones conflictivas
    # Nota: Con los datos simulados (4 filas) esto fallará por falta de datos, 
    # pero funcionará perfecto con tu dataset completo.
    # modelo_bertopic = descubrir_temas_explicitos(df_procesado, col_letra='letra')
    
    print("\n¡Tarea 2 finalizada con éxito!")